In [8]:
import pandas as pd
import yfinance as yf

In [9]:
path = "../data/SPX500 Original.xlsm"
df = pd.read_excel(path, sheet_name = "Price", index_col=0)
df

,MMM UN Equity,ABT UN Equity,2056Q UN Equity,AMD UN Equity,45200Q UN Equity,968734Q UN Equity,950967Q UN Equity,APD UN Equity,1281683D UN Equity,1312089D UN Equity,...,CBOE UF Equity,UAL UW Equity,WCG UN Equity,AMD UW Equity,FTNT UW Equity,LIN UN Equity,ROL UN Equity,JKHY UW Equity,KEYS UN Equity,REG UW Equity
data,3M Co,Abbott Laboratories,ACME-Cleveland Corp,Advanced Micro Devices Inc,AEP Texas Inc,Aeroquip-Vickers Inc,AIG Life Holdings Inc,Air Products & Chemicals Inc,Alberto-Culver Co,Albertson's LLC,...,Cboe Global Markets Inc,United Continental Holdings Inc,WellCare Health Plans Inc,Advanced Micro Devices Inc,Fortinet Inc,Linde PLC,Rollins Inc,Jack Henry & Associates Inc,Keysight Technologies Inc,Regency Centers Corp
1990-12-31 00:00:00,20.629,5.0348,4.75,2.4375,22,18.25,7.6875,12.6542,9.5408,18.25,...,NaN,NaN,NaN,NaN,NaN,NaN,1.2437,0.125,NaN,NaN
1991-01-31 00:00:00,20.419,4.881,5.375,3.5625,22,22,8.1875,13.6364,9.2897,19.375,...,NaN,NaN,NaN,NaN,NaN,NaN,1.2144,0.1667,NaN,NaN
1991-02-28 00:00:00,21.291,5.1887,6.625,4.0625,22,24.375,9.1875,14.8499,8.8234,19.25,...,NaN,NaN,NaN,NaN,NaN,NaN,1.3242,0.2292,NaN,NaN
1991-03-29 00:00:00,21.291,5.3705,6.625,5.1875,22.5625,22.5,9.625,15.6588,8.5723,23.4375,...,NaN,NaN,NaN,NaN,NaN,NaN,1.4047,0.25,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-07-31 00:00:00,212.32,65.54,30,18.34,20.9375,57.9375,44.55,164.17,37.49,25.67,...,97.19,23.63,267.42,18.33,62.91,167.5,36.6267,134.7,58,NaN
2018-08-31 00:00:00,210.92,66.84,30,25.2,20.9375,57.9375,44.55,166.29,37.49,25.67,...,100.77,23.63,302.57,25.17,83.76,158.19,40.0533,158.44,64.89,NaN
2018-09-28 00:00:00,210.71,73.36,30,30.885,20.9375,57.9375,44.55,167.05,37.49,25.67,...,95.96,89.06,320.49,30.89,92.27,160.73,40.46,160.08,66.28,NaN
2018-10-31 00:00:00,190.26,68.94,30,18.21,20.9375,57.9375,44.55,154.35,37.49,25.67,...,112.85,85.51,275.99,18.21,82.18,165.47,39.4667,149.83,57.08,NaN


In [10]:
df_mini = df.iloc[110:, 85:100]
df_mini = df_mini.apply(pd.to_numeric, errors="coerce")
df_mini.dtypes
#i dati vanno dal 31/1/2000 fino al 30/11/2018 di 100 titoli

CNP UN Equity         float64
NAE US Equity         float64
0226226D UN Equity    float64
140402Q UN Equity     float64
1073675D UQ Equity    float64
1327Q UN Equity       float64
CVX UN Equity         float64
2251Q UN Equity       float64
9876566D UN Equity    float64
CI UN Equity          float64
CCTYQ UN Equity       float64
2200Q US Equity       float64
C UN Equity           float64
CKL UN Equity         float64
CLX UN Equity         float64
dtype: object

In [11]:
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage

mu = mean_historical_return(df_mini,frequency=12)
S = CovarianceShrinkage(df_mini,frequency=12).ledoit_wolf()

print(mu)

CNP UN Equity         0.022167
NAE US Equity         0.000000
0226226D UN Equity    0.010564
140402Q UN Equity     0.011133
1073675D UQ Equity    0.004402
1327Q UN Equity       0.000000
CVX UN Equity         0.057121
2251Q UN Equity       0.000000
9876566D UN Equity    0.083454
CI UN Equity          0.125958
CCTYQ UN Equity      -0.157451
2200Q US Equity       0.000000
C UN Equity          -0.092211
CKL UN Equity         0.000000
CLX UN Equity         0.068267
dtype: float64


In [14]:
from pypfopt import objective_functions
from pypfopt import EfficientFrontier

ef = EfficientFrontier(mu, S)
ef.add_objective(objective_functions.L2_reg, gamma=0.1)
w = ef.max_sharpe()
print(ef.clean_weights())

OrderedDict({'CNP UN Equity': 0.0, 'NAE US Equity': 0.0, '0226226D UN Equity': 0.0, '140402Q UN Equity': 0.04139, '1073675D UQ Equity': 0.0, '1327Q UN Equity': 0.0, 'CVX UN Equity': 0.16709, '2251Q UN Equity': 0.0, '9876566D UN Equity': 0.25887, 'CI UN Equity': 0.29697, 'CCTYQ UN Equity': 0.0, '2200Q US Equity': 0.0, 'C UN Equity': 0.0, 'CKL UN Equity': 0.0, 'CLX UN Equity': 0.23568})


c:\Users\feder\AppData\Local\Programs\Python\Python313\Lib\site-packages\pypfopt\efficient_frontier\efficient_frontier.py:259: UserWarning: max_sharpe transforms the optimization problem so additional objectives may not work as expected.
  warnings.warn(


In [15]:
ef.portfolio_performance(verbose=True)

Expected annual return: 8.5%
Annual volatility: 14.4%
Sharpe Ratio: 0.59


(np.float64(0.08510393589177936),
 np.float64(0.14435706728535386),
 np.float64(0.5895377170800548))

In [16]:
from pypfopt.discrete_allocation import DiscreteAllocation, get_latest_prices

latest_prices = get_latest_prices(df_mini)
da = DiscreteAllocation(w, latest_prices, total_portfolio_value=20000)
allocation, leftover = da.lp_portfolio()
print(allocation)

{'140402Q UN Equity': 12, 'CVX UN Equity': 28, '9876566D UN Equity': 40, 'CI UN Equity': 27, 'CLX UN Equity': 28}
